# Stunting Prediction — Linear Regression Task
Predicting **Height-for-Age Z-Score** for toddlers (0-60 months) from gender,
height, weight-for-age z-score, and weight-for-height z-score.

Dataset: *Stunting and Nutritional Status of Toddlers — Jeneponto, Indonesia*
(Preprocessed / ML Compatible file), Mendeley Data, 40,071 records.

This notebook reads the source file directly from Google Drive — no manual
upload required after the first time you add the file to your Drive.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

## 1. Load data from Google Drive

**One-time setup:** upload `Preprocessed Data.xlsx` to your Google Drive
(anywhere — e.g. `My Drive/stunting_project/Preprocessed Data.xlsx`).
Update `DRIVE_PATH` below to match where you put it.

Running this cell will prompt you to authorize Colab's access to your Drive
the first time — after that, this notebook will always read the file
straight from Drive with no upload dialog.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS to match the actual path of the file in your Google Drive
DRIVE_PATH = '/content/drive/MyDrive/stunting_project/Preprocessed Data.xlsx'

df = pd.read_excel(DRIVE_PATH)

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:\n", df.dtypes)
df.head()

## 2. Data Cleaning

Issues identified during exploration:
- `Weight` is corrupted: ~50% of values were auto-converted to dates by
  Excel (e.g. `2.8` -> Feb 8) before this file was ever exported. This is
  unrecoverable and the column is dropped (redundant anyway — we already
  have `Z-Score W/A` and `Z-Score W/H`, which summarize weight in a cleaner,
  WHO-standardized form).
- `Z-Score  W/A`, `Z-Score H/A`, `Z-Score W/H` are stored as text with
  comma decimal separators (e.g. `"-1,94"`) and must be converted to numeric.
- `Height for Age` is a direct threshold classification of our target
  `Z-Score H/A` (cutoff exactly at -2 SD) — this is **target leakage** and
  must be dropped.
- `Weight for Age` and `Weight for Height` are similarly just binned/coded
  versions of `Z-Score W/A` and `Z-Score W/H` — redundant, dropped.
- A small number of rows contain biologically implausible z-scores
  (|z| > 6, up to several thousand — clear data-entry errors) and are
  dropped.

In [ ]:
df_clean = df.copy()

# --- Fix comma-decimal numeric columns ---
comma_cols = ['Weight', 'Z-Score  W/A', 'Z-Score H/A', 'Z-Score W/H']
for col in comma_cols:
    df_clean[col] = (
        df_clean[col].astype(str).str.strip().str.replace(',', '.', regex=False)
    )
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# --- Drop leakage, redundant, and corrupted columns ---
df_clean = df_clean.drop(columns=[
    'No.', 'Weight', 'Height for Age', 'Weight for Age', 'Weight for Height'
])

# --- Drop rows missing target or predictor z-scores ---
before = len(df_clean)
df_clean = df_clean.dropna(subset=['Z-Score H/A', 'Z-Score  W/A', 'Z-Score W/H', 'Age (Month)'])
print(f"Dropped {before - len(df_clean)} rows with missing values")

# --- Drop biologically implausible outliers (|z| > 6) ---
before = len(df_clean)
df_clean = df_clean[
    (df_clean['Z-Score  W/A'].abs() <= 6) &
    (df_clean['Z-Score H/A'].abs() <= 6) &
    (df_clean['Z-Score W/H'].abs() <= 6)
]
print(f"Dropped {before - len(df_clean)} rows with implausible outlier z-scores (|z|>6)")

df_clean = df_clean.rename(columns={'Z-Score  W/A': 'Z-Score W/A'})
df_clean = df_clean.reset_index(drop=True)

print(f"\nFinal cleaned shape: {df_clean.shape}")
df_clean.describe()

## 3. Exploratory Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_clean['Z-Score H/A'], bins=50, kde=True, ax=axes[0])
axes[0].axvline(-2, color='red', linestyle='--', label='Stunting threshold (-2 SD)')
axes[0].set_title('Distribution of Height-for-Age Z-Score (target)')
axes[0].legend()

sns.boxplot(y=df_clean['Z-Score H/A'], ax=axes[1])
axes[1].set_title('Z-Score H/A — spread & outliers')
plt.tight_layout()
plt.show()

print(df_clean['Z-Score H/A'].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df_clean['Age (Month)'], bins=30, ax=axes[0,0])
axes[0,0].set_title('Age (Months) distribution')

sns.histplot(df_clean['Height'], bins=40, ax=axes[0,1])
axes[0,1].set_title('Height (cm) distribution')

sns.countplot(x='Gender', data=df_clean, ax=axes[1,0])
axes[1,0].set_title('Gender distribution (0/1)')

sns.histplot(df_clean['Z-Score W/A'], bins=40, ax=axes[1,1])
axes[1,1].set_title('Z-Score W/A distribution')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,6))
corr = df_clean.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix')
plt.show()

print(corr['Z-Score H/A'].sort_values(ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Age (Month)', 'Height', 'Z-Score W/A']):
    ax.scatter(df_clean[col], df_clean['Z-Score H/A'], alpha=0.1, s=8)
    ax.set_xlabel(col)
    ax.set_ylabel('Z-Score H/A')
    ax.set_title(f'{col} vs Z-Score H/A')
plt.tight_layout()
plt.show()

## 4. Feature Engineering — Interpretation

| Feature | Decision | Justification |
|---|---|---|
| `Age (Month)` | **Drop** | Near-zero correlation with target (-0.02) and severe multicollinearity with `Height` (r=0.93). WHO z-scores are already age-standardized, so age's signal is absorbed by height. |
| `Height` | **Keep** | Structural predictor (r=0.28); absorbs the age-related signal without the redundancy. |
| `Z-Score W/A` | **Keep — highest expected weight** | Strongest correlate (r=0.64) — chronic undernutrition affects weight and height jointly. |
| `Z-Score W/H` | **Keep** | Weak individually (r=-0.11) but not redundant with W/A (r=0.68 between them) — captures the "stunted vs. wasted" distinction documented in nutrition literature. |
| `Gender` | **Keep** | Weak (r=-0.07) but free (already binary), consistent with known sex differences in stunting risk. |

Note: `Height`'s correlation with the target is expected to be partly
mechanical, since WHO Height-for-Age z-scores are a deterministic nonlinear
function of height, age, and sex via WHO growth-standard lookup tables. This
is a legitimate regression problem, but it explains why tree-based models
are expected to outperform linear ones — the true relationship is
nonlinear.

## 5. Encoding, Split, and Standardization

In [ ]:
features = ['Gender', 'Height', 'Z-Score W/A', 'Z-Score W/H']
target = 'Z-Score H/A'

X = df_clean[features].copy()
y = df_clean[target].copy()

# Gender is already binary numeric (0/1) — no further encoding needed
print("Gender unique values:", X['Gender'].unique())
print(X.dtypes)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=features, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=features, index=X_test.index)

X_train_scaled.describe()

## 6. Model 1 — SGD Regressor (Gradient Descent) with Loss Curve

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score

sgd = SGDRegressor(
    loss='squared_error',
    penalty='l2',
    alpha=0.0001,
    learning_rate='invscaling',
    eta0=0.01,
    max_iter=1,
    warm_start=True,
    random_state=42
)

n_epochs = 100
train_losses, test_losses = [], []

for epoch in range(n_epochs):
    sgd.partial_fit(X_train_scaled, y_train)
    train_pred = sgd.predict(X_train_scaled)
    test_pred = sgd.predict(X_test_scaled)
    train_losses.append(mean_squared_error(y_train, train_pred))
    test_losses.append(mean_squared_error(y_test, test_pred))

print(f"Final Train MSE: {train_losses[-1]:.4f}")
print(f"Final Test MSE: {test_losses[-1]:.4f}")
print(f"Test R2: {r2_score(y_test, sgd.predict(X_test_scaled)):.4f}")

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_losses, label='Train Loss (MSE)')
plt.plot(test_losses, label='Test Loss (MSE)')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('SGDRegressor: Loss Curve (Train vs Test)')
plt.legend()
plt.grid(True)
plt.show()

## 7. Comparison Models — Linear Regression, Random Forest, Decision Tree

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

models = {
    'SGD Regressor': sgd,
    'Linear Regression (OLS)': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'Decision Tree': DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
}

results = {}

for name, model in models.items():
    if name != 'SGD Regressor':  # already trained above
        model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    results[name] = {
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2': r2_score(y_test, pred)
    }

results_df = pd.DataFrame(results).T
print(results_df.sort_values('RMSE'))

### Discussion — Performance, Dataset Impact, and Model Justification

**Fill in exact numbers from the table above, then use this structure:**

1. **Performance via loss metric (RMSE/MSE):** Rank the five models by RMSE.
   Random Forest is expected to have the lowest error, followed closely by
   Decision Tree, with the linear models (SGD, OLS, Ridge) close behind but
   slightly higher. Explain what the RMSE gap means in real terms — e.g. an
   RMSE of 0.24 means predictions are off by about 0.24 z-score units on
   average.

2. **How the dataset impacted performance:** WHO Height-for-Age z-scores are
   computed via a nonlinear lookup-table transformation of raw height, age,
   and sex — not a linear formula. This structurally favors tree-based
   models (Decision Tree, Random Forest), which can carve up the feature
   space into nonlinear regions, over linear models, which must approximate
   that curve with a straight hyperplane. The linear models still perform
   well (R² > 0.95) because `Z-Score W/A` is strongly and fairly linearly
   correlated with the target.

3. **Model justification:** Random Forest is selected as the production
   model because it achieves the best raw error metrics, is robust to the
   outliers observed during cleaning, and does not require the linearity
   assumption to hold — appropriate given the known nonlinear structure of
   WHO z-score calculations.

## 8. Before/After Scatter Plots

In [ ]:
simple_lr = LinearRegression()
simple_lr.fit(X_train_scaled[['Z-Score W/A']], y_train)

x_range = np.linspace(X_train_scaled['Z-Score W/A'].min(), X_train_scaled['Z-Score W/A'].max(), 100).reshape(-1,1)
y_line = simple_lr.predict(x_range)

fig, axes = plt.subplots(1, 2, figsize=(13,5))

axes[0].scatter(X_train_scaled['Z-Score W/A'], y_train, alpha=0.1, s=8, color='steelblue')
axes[0].set_title('Before: Z-Score W/A vs Z-Score H/A (raw)')
axes[0].set_xlabel('Z-Score W/A (standardized)')
axes[0].set_ylabel('Z-Score H/A')

axes[1].scatter(X_train_scaled['Z-Score W/A'], y_train, alpha=0.1, s=8, color='steelblue')
axes[1].plot(x_range, y_line, color='red', linewidth=2,
             label=f'Fitted line (R2={simple_lr.score(X_train_scaled[["Z-Score W/A"]], y_train):.3f})')
axes[1].set_title('After: With Fitted Regression Line')
axes[1].set_xlabel('Z-Score W/A (standardized)')
axes[1].set_ylabel('Z-Score H/A')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_model = models['Random Forest']
y_pred_best = best_model.predict(X_test_scaled)

plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_best, alpha=0.15, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual Z-Score H/A')
plt.ylabel('Predicted Z-Score H/A')
plt.title('Random Forest: Actual vs Predicted')
plt.legend()
plt.axis('equal')
plt.show()

## 9. Save the Best Model

Saves to Google Drive (not a browser download) so the files persist and are
easy to pull into the FastAPI project without re-uploading anything.

In [ ]:
import joblib
import os

SAVE_DIR = '/content/drive/MyDrive/stunting_project/model_artifacts'
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(best_model, os.path.join(SAVE_DIR, 'best_model_random_forest.pkl'))
joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler.pkl'))
joblib.dump(features, os.path.join(SAVE_DIR, 'feature_names.pkl'))

print(f"Saved to Google Drive at: {SAVE_DIR}")
print(os.listdir(SAVE_DIR))